# Bronze 원천·결과 미리보기

프로젝트 `.venv`를 커널로 선택하세요. `config.py`의 입력·출력·시트·컬럼 설정을 사용합니다.
원천은 읽기만 하며 표준화를 수행하지 않습니다. Bronze 생성: `python src/01_bronze.py`.
수집 이력의 일시는 `YYYY-MM-DD HH:MI:SS`, KST입니다.

In [6]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# 루트 또는 src에서 실행 가능. 다른 위치에서는 이 값을 프로젝트 경로로 지정하세요.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src/config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT에 프로젝트 루트 경로를 지정하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import config
importlib.reload(config)
CONFIG = config.CONFIG
spec = importlib.util.spec_from_file_location("bronze_preview", PROJECT_ROOT / "src/01_bronze.py")
bronze = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bronze)
print("입력:", bronze.absolute(CONFIG.source_root))
print("출력:", bronze.absolute(CONFIG.output_root))

입력: D:\IT\mlo3_p2_team3_EB\data\00_origin
출력: D:\IT\mlo3_p2_team3_EB\data\01_bronze


## 설정에 해당하는 파일 경로

In [7]:
jobs = bronze.plan(CONFIG)
paths = pd.DataFrame([{"source_path": str(source), "sheet_name": sheet,
                       "parquet_path": str(destination), "exists": destination.is_file()}
                      for source, rule, sheet, destination in jobs])
display(paths)

,source_path,sheet_name,parquet_path,exists
0,D:\IT\mlo3_p2_team3_EB\data\00_origin\event\ra...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\event\ra...,True
1,D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR...,True
2,D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR...,True
3,D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\TR...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\TR...,True
4,D:\IT\mlo3_p2_team3_EB\data\00_origin\member\M...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\member\M...,True
5,D:\IT\mlo3_p2_team3_EB\data\00_origin\order\be...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\order\be...,True
6,D:\IT\mlo3_p2_team3_EB\data\00_origin\order\be...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\order\be...,True
7,D:\IT\mlo3_p2_team3_EB\data\00_origin\race\RC_...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\race\RC_...,True
8,D:\IT\mlo3_p2_team3_EB\data\00_origin\race\RC_...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\race\RC_...,True
9,D:\IT\mlo3_p2_team3_EB\data\00_origin\race\RC_...,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\race\RC_...,True


## 원천 파일별 head(3)
자동 타입 추론 없이 동일한 원천 리더로 확인합니다. 대용량 원천도 전체를 읽으므로 메모리가 필요합니다.

In [8]:
from dataclasses import replace
for source, rule, sheet, destination in jobs:
    print(f"\n원천: {source} | 시트: {sheet}")
    effective = replace(rule, sheet_names=(sheet,)) if sheet is not None else rule
    _, table = next(bronze.read_tables(source, effective))
    display(table.slice(0, 3).to_pandas().head(3))
    del table


원천: D:\IT\mlo3_p2_team3_EB\data\00_origin\event\race_change_events.jsonl | 시트: None


,eventType,announcedAt,trackRaw,race,horse,change,reason
0,TRACK_STATUS,20260522 23:58,다습 ( 14.0%),None,None,None,NaN
1,TRACK_STATUS,20260523 23:58,다습 ( 12.0%),None,None,None,NaN
2,JOCKEY_CHANGE,20260523 23:58,NaN,"{'rcDate': '2026/05/23', 'rcNo': '1'}","{'chulNo': '6', 'hrName': '원평가속도'}","{'before': '나유나', 'after': '임재광', 'budamRaw': ...",NaN



원천: D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR_VET_HIST.csv | 시트: None


,VET_DT_RAW,HR_NAME,TR_GRP_NO,VET_PLACE_NM,VET_NM_RAW,REPORT_DT
0,2026.05.16,대방가,05,KRA 제주경마,우전지파행,20260520
1,2026.05.16,사라호,19,KRA 제주경마,좌전지파행,20260520
2,2026.05.16,색달미소,06,KRA 제주경마,기타 파행,20260520



원천: D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR_WEEKLY_SNAPSHOT.csv | 시트: None


,SNAPSHOT_DT,HR_NAME,PRD_NM,SEX_NM,BIRTH_RAW,AGE_RAW,GRADE_RAW,TR_GRP_NO,TR_NAME,OW_NAME,...,TOT_3RD,Y1_RC_CNT,Y1_1ST,Y1_2ND,Y1_3RD,PRIZE_AMT_RAW,RATING_RAW,INTRO_PRICE_RAW,SIRE_NAME,DAM_NAME
0,20260520,가라공주,제,암,2022/05/15,4,제6,88,외부마,조종호,...,0,5,0,1,0,3220000,,"3,000천원(개별)",화룡특급,희망여신
1,20260520,가문의여신,제,암,2023/03/24,3,제6,88,외부마,김영구,...,0,2,0,0,0,700000,,"15,000천원(개별)",오라태양,가문의전설
2,20260520,가속의힘,제,암,2023/04/24,3,미,08,문성호,고영조,...,0,0,0,0,0,0,,자가생산,일대태수,세계의보물


... (14 outputs left out)


원천: D:\IT\mlo3_p2_team3_EB\data\00_origin\race\RC_RESULT.csv | 시트: None


,RC_DATE,RC_NO,ORD_RAW,CHUL_NO,HR_NAME,WG_HR_RAW,RC_TIME_RAW,MARGIN_RAW,PASS_ORD_RAW
0,20260522,1,1,6,센돌이,279+14,1:13.2,,1- - - 1- 1- 1
1,20260522,1,2,10,전설마루,285-3,1:13.3,목,6- - - 6- 6- 6
2,20260522,1,3,8,달의울림,260+1,1:14.0,4,3- - - 4- 4- 2



원천: D:\IT\mlo3_p2_team3_EB\data\00_origin\race\RC_SALES.csv | 시트: None


,RC_DATE,RC_NO,POOL_NM,SALES_AMT_RAW
0,20260522,1,단식,"12,548,000"
1,20260522,1,연식,"24,207,200"
2,20260522,1,복식,"289,459,600"


## Bronze 파일별 head(3)

In [9]:
for source, rule, sheet, destination in jobs:
    print(f"\nBronze: {destination}")
    if not destination.is_file():
        print("아직 생성되지 않았습니다. 먼저 01_bronze.py를 실행하세요.")
        continue
    # 첫 row group만 읽어 전체 파일 로드를 줄입니다.
    parquet = pq.ParquetFile(destination)
    if parquet.num_row_groups:
        preview = parquet.read_row_group(0).slice(0, 3).to_pandas()
    else:
        preview = pq.read_table(destination).slice(0, 3).to_pandas()
    parquet.close()
    display(preview.head(3))


Bronze: D:\IT\mlo3_p2_team3_EB\data\01_bronze\event\race_change_events.parquet


,eventType,announcedAt,trackRaw,race,horse,change,reason,source_file,ingested_at
0,TRACK_STATUS,20260522 23:58,다습 ( 14.0%),None,None,None,NaN,race_change_events.jsonl,2026-11-20 14:40:35
1,TRACK_STATUS,20260523 23:58,다습 ( 12.0%),None,None,None,NaN,race_change_events.jsonl,2026-11-20 14:40:35
2,JOCKEY_CHANGE,20260523 23:58,NaN,"{'rcDate': '2026/05/23', 'rcNo': '1'}","{'chulNo': '6', 'hrName': '원평가속도'}","{'before': '나유나', 'after': '임재광', 'budamRaw': ...",NaN,race_change_events.jsonl,2026-11-20 14:40:35



Bronze: D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR_VET_HIST.parquet


,VET_DT_RAW,HR_NAME,TR_GRP_NO,VET_PLACE_NM,VET_NM_RAW,REPORT_DT,source_file,ingested_at
0,2026.05.16,대방가,05,KRA 제주경마,우전지파행,20260520,HR_VET_HIST.csv,2026-11-20 14:40:35
1,2026.05.16,사라호,19,KRA 제주경마,좌전지파행,20260520,HR_VET_HIST.csv,2026-11-20 14:40:35
2,2026.05.16,색달미소,06,KRA 제주경마,기타 파행,20260520,HR_VET_HIST.csv,2026-11-20 14:40:35



Bronze: D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR_WEEKLY_SNAPSHOT.parquet


,SNAPSHOT_DT,HR_NAME,PRD_NM,SEX_NM,BIRTH_RAW,AGE_RAW,GRADE_RAW,TR_GRP_NO,TR_NAME,OW_NAME,...,Y1_1ST,Y1_2ND,Y1_3RD,PRIZE_AMT_RAW,RATING_RAW,INTRO_PRICE_RAW,SIRE_NAME,DAM_NAME,source_file,ingested_at
0,20260520,가라공주,제,암,2022/05/15,4,제6,88,외부마,조종호,...,0,1,0,3220000,,"3,000천원(개별)",화룡특급,희망여신,HR_WEEKLY_SNAPSHOT.csv,2026-11-20 14:40:35
1,20260520,가문의여신,제,암,2023/03/24,3,제6,88,외부마,김영구,...,0,0,0,700000,,"15,000천원(개별)",오라태양,가문의전설,HR_WEEKLY_SNAPSHOT.csv,2026-11-20 14:40:35
2,20260520,가속의힘,제,암,2023/04/24,3,미,08,문성호,고영조,...,0,0,0,0,,자가생산,일대태수,세계의보물,HR_WEEKLY_SNAPSHOT.csv,2026-11-20 14:40:35


... (14 outputs left out)


Bronze: D:\IT\mlo3_p2_team3_EB\data\01_bronze\race\RC_RESULT.parquet


,RC_DATE,RC_NO,ORD_RAW,CHUL_NO,HR_NAME,WG_HR_RAW,RC_TIME_RAW,MARGIN_RAW,PASS_ORD_RAW,source_file,ingested_at
0,20260522,1,1,6,센돌이,279+14,1:13.2,,1- - - 1- 1- 1,RC_RESULT.csv,2026-11-20 14:40:37
1,20260522,1,2,10,전설마루,285-3,1:13.3,목,6- - - 6- 6- 6,RC_RESULT.csv,2026-11-20 14:40:37
2,20260522,1,3,8,달의울림,260+1,1:14.0,4,3- - - 4- 4- 2,RC_RESULT.csv,2026-11-20 14:40:37



Bronze: D:\IT\mlo3_p2_team3_EB\data\01_bronze\race\RC_SALES.parquet


,RC_DATE,RC_NO,POOL_NM,SALES_AMT_RAW,source_file,ingested_at
0,20260522,1,단식,"12,548,000",RC_SALES.csv,2026-11-20 14:40:37
1,20260522,1,연식,"24,207,200",RC_SALES.csv,2026-11-20 14:40:37
2,20260522,1,복식,"289,459,600",RC_SALES.csv,2026-11-20 14:40:37


## 수집 이력과 최근 실행 대사

In [10]:
manifest_path = bronze.absolute(CONFIG.manifest_path)
print("수집 이력:", manifest_path)
if manifest_path.exists():
    records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line]
    history = pd.DataFrame(records)
    display(history.head(3))
    latest = history[history["run_id"] == history.iloc[-1]["run_id"]]
    columns = ["source_relative_path", "sheet_name", "ingested_at", "saved_at", "timezone",
               "source_rows", "bronze_rows", "row_count_match", "values_match", "status"]
    display(latest.reindex(columns=columns))
    assert not latest["status"].eq("failed").any(), "실패한 파일이 있습니다. 수집 이력을 확인하세요."
    assert latest["row_count_match"].eq(True).all()
    assert latest["values_match"].eq(True).all()
    print("최근 실행: 파일/시트", len(latest), "개, 원천 행 수", int(latest["source_rows"].sum()),
          "= Bronze 행 수", int(latest["bronze_rows"].sum()))
else:
    print("수집 이력이 없습니다. 먼저 01_bronze.py를 실행하세요.")

수집 이력: D:\IT\mlo3_p2_team3_EB\meta\bronze_ingestion.jsonl


,run_id,source_path,source_relative_path,sheet_name,output_path,timezone,utc_offset,ingested_at,selected_columns,source_sha256,...,saved_at,bronze_rows,row_count_match,values_match,output_sha256,output_columns,data_ingested_at,provenance_columns,failed_at,error
0,51a87492-10d5-4156-9e6a-c0d4a6df84f7,D:\IT\mlo3_p2_team3_EB\data\00_origin\event\ra...,event/race_change_events.jsonl,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\event\ra...,KST,+09:00,2026-11-20 14:40:35,None,c589dfe8e4681e320f0a7ba490ed631cbd39629c7e42b6...,...,2026-11-20 14:40:35,149.0,True,True,c4515186fd03482170fe78965f1a2ee7b12e17855684d5...,"[eventType, announcedAt, trackRaw, race, horse...",NaN,NaN,NaN,NaN
1,51a87492-10d5-4156-9e6a-c0d4a6df84f7,D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR...,horse/HR_VET_HIST.csv,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR...,KST,+09:00,2026-11-20 14:40:35,None,75ff0fd6ca1736bcc83d7fb1b09429217d2d593443dbfe...,...,2026-11-20 14:40:35,1488.0,True,True,f3497fe67b40488d9d37a264fbaa08311873a5ae38cc36...,"[VET_DT_RAW, HR_NAME, TR_GRP_NO, VET_PLACE_NM,...",NaN,NaN,NaN,NaN
2,51a87492-10d5-4156-9e6a-c0d4a6df84f7,D:\IT\mlo3_p2_team3_EB\data\00_origin\horse\HR...,horse/HR_WEEKLY_SNAPSHOT.csv,None,D:\IT\mlo3_p2_team3_EB\data\01_bronze\horse\HR...,KST,+09:00,2026-11-20 14:40:35,None,5d20204a142c02780f8f74cda8604e90ea3200deb9633f...,...,2026-11-20 14:40:35,52642.0,True,True,e4f3d0b427f15586cdd512a3adb0cc99b6827b06328743...,"[SNAPSHOT_DT, HR_NAME, PRD_NM, SEX_NM, BIRTH_R...",NaN,NaN,NaN,NaN


,source_relative_path,sheet_name,ingested_at,saved_at,timezone,source_rows,bronze_rows,row_count_match,values_match,status
84,event/race_change_events.jsonl,None,2026-11-20 14:58:13,2026-11-20 14:58:13,KST,149,149.0,True,True,unchanged
85,horse/HR_VET_HIST.csv,None,2026-11-20 14:58:13,2026-11-20 14:58:13,KST,1488,1488.0,True,True,unchanged
86,horse/HR_WEEKLY_SNAPSHOT.csv,None,2026-11-20 14:58:13,2026-11-20 14:58:14,KST,52642,52642.0,True,True,unchanged
87,horse/TR_WEEKLY_SNAPSHOT.csv,None,2026-11-20 14:58:14,2026-11-20 14:58:14,KST,954,954.0,True,True,unchanged
88,member/MBR_MASTER.csv,None,2026-11-20 14:58:14,2026-11-20 14:58:14,KST,4383,4383.0,True,True,unchanged
89,order/bet_order.csv,None,2026-11-20 14:58:14,2026-11-20 14:58:14,KST,27907,27907.0,True,True,unchanged
90,order/bet_ticket.csv,None,2026-11-20 14:58:14,2026-11-20 14:58:15,KST,523416,523416.0,True,True,unchanged
91,race/RC_ENTRY.csv,None,2026-11-20 14:58:15,2026-11-20 14:58:15,KST,3666,3666.0,True,True,unchanged
92,race/RC_ODDS.csv,None,2026-11-20 14:58:15,2026-11-20 14:58:15,KST,3666,3666.0,True,True,unchanged
93,race/RC_RACE.csv,None,2026-11-20 14:58:15,2026-11-20 14:58:16,KST,386,386.0,True,True,unchanged


최근 실행: 파일/시트 12 개, 원천 행 수 625411 = Bronze 행 수 625411
